In [1]:
import pandas as pd

games = pd.read_csv("games.csv")
games_details = pd.read_csv("games_details.csv")
ranking = pd.read_csv("ranking.csv")

/var/folders/2r/46mxdql52bn78bp6pr2cps340000gn/T/ipykernel_40182/3465009782.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  games_details = pd.read_csv("games_details.csv")


In [2]:
games.head()

,GAME_DATE_EST,GAME_ID,GAME_STATUS_TEXT,HOME_TEAM_ID,VISITOR_TEAM_ID,SEASON,TEAM_ID_home,PTS_home,FG_PCT_home,FT_PCT_home,...,AST_home,REB_home,TEAM_ID_away,PTS_away,FG_PCT_away,FT_PCT_away,FG3_PCT_away,AST_away,REB_away,HOME_TEAM_WINS
0,2022-12-22,22200477,Final,1610612740,1610612759,2022,1610612740,126.0,0.484,0.926,...,25.0,46.0,1610612759,117.0,0.478,0.815,0.321,23.0,44.0,1
1,2022-12-22,22200478,Final,1610612762,1610612764,2022,1610612762,120.0,0.488,0.952,...,16.0,40.0,1610612764,112.0,0.561,0.765,0.333,20.0,37.0,1
2,2022-12-21,22200466,Final,1610612739,1610612749,2022,1610612739,114.0,0.482,0.786,...,22.0,37.0,1610612749,106.0,0.470,0.682,0.433,20.0,46.0,1
3,2022-12-21,22200467,Final,1610612755,1610612765,2022,1610612755,113.0,0.441,0.909,...,27.0,49.0,1610612765,93.0,0.392,0.735,0.261,15.0,46.0,1
4,2022-12-21,22200468,Final,1610612737,1610612741,2022,1610612737,108.0,0.429,1.000,...,22.0,47.0,1610612741,110.0,0.500,0.773,0.292,20.0,47.0,0


In [65]:
# Keep only columns we need at first
base = games[
    [
        "GAME_ID",
        "GAME_DATE_EST",
        "HOME_TEAM_ID",
        "VISITOR_TEAM_ID",
        "PTS_home",
        "PTS_away",
    ]
].copy()

# Parse date and sort chronologically
base["GAME_DATE_EST"] = pd.to_datetime(base["GAME_DATE_EST"])
base = base.sort_values(["GAME_DATE_EST", "GAME_ID"]).reset_index(drop=True)

base = base.dropna(subset=["PTS_home", "PTS_away"]).copy()
base = base.drop_duplicates(subset=["GAME_ID"]).copy()

# Create target
base["home_team_win"] = (base["PTS_home"] > base["PTS_away"]).astype(int)

# Optional sanity checks
print("Base shape:", base.shape)
base.head()


Base shape: (26523, 7)


,GAME_ID,GAME_DATE_EST,HOME_TEAM_ID,VISITOR_TEAM_ID,PTS_home,PTS_away,home_team_win
0,10300001,2003-10-05,1610612762,1610612742,90.0,85.0,1
1,10300002,2003-10-06,1610612763,1610612749,105.0,94.0,1
2,10300003,2003-10-07,1610612765,1610612739,96.0,100.0,0
3,10300004,2003-10-07,1610612742,1610612753,99.0,89.0,1
4,10300005,2003-10-07,1610612757,1610612745,104.0,80.0,1


In [68]:
home = base[
    ["GAME_ID", "GAME_DATE_EST", "HOME_TEAM_ID", "PTS_home", "PTS_away", "home_team_win"]
].copy()
home.columns = ["GAME_ID", "date", "team_id", "points_for", "points_against", "win"]

away = base[
    ["GAME_ID", "GAME_DATE_EST", "VISITOR_TEAM_ID", "PTS_away", "PTS_home", "home_team_win"]
].copy()
away.columns = ["GAME_ID", "date", "team_id", "points_for", "points_against", "win"]
away["win"] = 1 - away["win"]

team_games = pd.concat([home, away], ignore_index=True)
team_games = team_games.sort_values(["team_id", "date", "GAME_ID"]).reset_index(drop=True)

# Derived per-team-game stat
team_games["point_diff"] = team_games["points_for"] - team_games["points_against"]

print("Team-games shape:", team_games.shape)
team_games.head()

Team-games shape: (53046, 7)


,GAME_ID,date,team_id,points_for,points_against,win,point_diff
0,10300011,2003-10-08,1610612737,80.0,83.0,0,-3.0
1,20300006,2003-10-29,1610612737,83.0,88.0,0,-5.0
2,20300024,2003-10-31,1610612737,94.0,100.0,0,-6.0
3,20300029,2003-11-01,1610612737,99.0,103.0,0,-4.0
4,20300042,2003-11-03,1610612737,90.0,80.0,1,10.0


In [69]:
team_games["rolling_win_pct_5"] = (
    team_games.groupby("team_id")["win"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games["rolling_win_pct_10"] = (
    team_games.groupby("team_id")["win"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

team_games["rolling_point_diff_5"] = (
    team_games.groupby("team_id")["point_diff"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

team_games["rolling_point_diff_10"] = (
    team_games.groupby("team_id")["point_diff"]
    .transform(lambda x: x.shift(1).rolling(10).mean())
)

In [70]:
team_games["prev_game_date"] = team_games.groupby("team_id")["date"].shift(1)
team_games["rest_days"] = (team_games["date"] - team_games["prev_game_date"]).dt.days

In [71]:
INITIAL_ELO = 1500
K = 20
HOME_ADVANTAGE = 100

def expected_win_prob(rating_a, rating_b):
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))

# Need game-level order for sequential ELO updates
elo_games = base.sort_values(["GAME_DATE_EST", "GAME_ID"]).reset_index(drop=True).copy()

elo_ratings = {}
home_elos = []
away_elos = []

for _, row in elo_games.iterrows():
    home_team = row["HOME_TEAM_ID"]
    away_team = row["VISITOR_TEAM_ID"]
    home_win = row["home_team_win"]

    # Ratings entering the game
    home_elo = elo_ratings.get(home_team, INITIAL_ELO)
    away_elo = elo_ratings.get(away_team, INITIAL_ELO)

    # Store pre-game ELOs
    home_elos.append(home_elo)
    away_elos.append(away_elo)

    # Apply home advantage only to expectation step
    home_elo_for_prob = home_elo + HOME_ADVANTAGE

    p_home = expected_win_prob(home_elo_for_prob, away_elo)
    p_away = 1 - p_home

    # Actual outcomes
    s_home = home_win
    s_away = 1 - home_win

    # Update ratings after the game
    elo_ratings[home_team] = home_elo + K * (s_home - p_home)
    elo_ratings[away_team] = away_elo + K * (s_away - p_away)

elo_games["home_elo"] = home_elos
elo_games["away_elo"] = away_elos
elo_games["elo_diff"] = elo_games["home_elo"] - elo_games["away_elo"]

# Keep only ELO columns needed for merge
elo_features = elo_games[["GAME_ID", "home_elo", "away_elo", "elo_diff"]].copy()


In [72]:
home_features = team_games[
    [
        "GAME_ID",
        "team_id",
        "rolling_win_pct_5",
        "rolling_win_pct_10",
        "rolling_point_diff_5",
        "rolling_point_diff_10",
        "rest_days",
    ]
].copy()

home_features = home_features.rename(
    columns={
        "team_id": "HOME_TEAM_ID",
        "rolling_win_pct_5": "home_rolling_win_pct_5",
        "rolling_win_pct_10": "home_rolling_win_pct_10",
        "rolling_point_diff_5": "home_rolling_point_diff_5",
        "rolling_point_diff_10": "home_rolling_point_diff_10",
        "rest_days": "home_rest_days",
    }
)

away_features = team_games[
    [
        "GAME_ID",
        "team_id",
        "rolling_win_pct_5",
        "rolling_win_pct_10",
        "rolling_point_diff_5",
        "rolling_point_diff_10",
        "rest_days",
    ]
].copy()

away_features = away_features.rename(
    columns={
        "team_id": "VISITOR_TEAM_ID",
        "rolling_win_pct_5": "away_rolling_win_pct_5",
        "rolling_win_pct_10": "away_rolling_win_pct_10",
        "rolling_point_diff_5": "away_rolling_point_diff_5",
        "rolling_point_diff_10": "away_rolling_point_diff_10",
        "rest_days": "away_rest_days",
    }
)


In [73]:
df = base.merge(
    home_features,
    on=["GAME_ID", "HOME_TEAM_ID"],
    how="left",
    validate="one_to_one",
)

df = df.merge(
    away_features,
    on=["GAME_ID", "VISITOR_TEAM_ID"],
    how="left",
    validate="one_to_one",
)

df = df.merge(
    elo_features,
    on="GAME_ID",
    how="left",
    validate="one_to_one",
)

print("Merged df shape:", df.shape)

Merged df shape: (26523, 20)


In [76]:
df

,GAME_ID,GAME_DATE_EST,HOME_TEAM_ID,VISITOR_TEAM_ID,PTS_home,PTS_away,home_team_win,home_rolling_win_pct_5,home_rolling_win_pct_10,home_rolling_point_diff_5,...,away_rolling_point_diff_10,away_rest_days,home_elo,away_elo,elo_diff,rolling_win_pct_5_diff,rolling_win_pct_10_diff,rolling_point_diff_5_diff,rolling_point_diff_10_diff,rest_days_diff
0,10300001,2003-10-05,1610612762,1610612742,90.0,85.0,1,NaN,NaN,NaN,...,NaN,NaN,1500.000000,1500.000000,0.000000,NaN,NaN,NaN,NaN,NaN
1,10300002,2003-10-06,1610612763,1610612749,105.0,94.0,1,NaN,NaN,NaN,...,NaN,NaN,1500.000000,1500.000000,0.000000,NaN,NaN,NaN,NaN,NaN
2,10300003,2003-10-07,1610612765,1610612739,96.0,100.0,0,NaN,NaN,NaN,...,NaN,NaN,1500.000000,1500.000000,0.000000,NaN,NaN,NaN,NaN,NaN
3,10300004,2003-10-07,1610612742,1610612753,99.0,89.0,1,NaN,NaN,NaN,...,NaN,NaN,1492.801300,1500.000000,-7.198700,NaN,NaN,NaN,NaN,NaN
4,10300005,2003-10-07,1610612757,1610612745,104.0,80.0,1,NaN,NaN,NaN,...,NaN,NaN,1500.000000,1500.000000,0.000000,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26518,22200474,2022-12-21,1610612760,1610612757,101.0,98.0,1,0.4,0.5,-0.6,...,3.4,2.0,1416.525781,1439.549888,-23.024107,-0.2,-0.1,-6.0,-4.4,0.0
26519,22200475,2022-12-21,1610612758,1610612747,134.0,120.0,1,0.4,0.6,-5.8,...,-2.6,2.0,1514.165040,1433.126182,81.038858,-0.2,0.1,-5.2,5.4,0.0
26520,22200476,2022-12-21,1610612746,1610612766,126.0,105.0,1,0.8,0.5,6.2,...,-7.7,2.0,1508.311444,1372.815989,135.495456,0.6,0.3,14.6,6.0,2.0
26521,22200477,2022-12-22,1610612740,1610612759,126.0,117.0,1,0.2,0.6,-6.4,...,-6.2,3.0,1563.074451,1379.702081,183.372370,-0.4,0.2,-6.0,9.8,0.0


In [75]:
df["rolling_win_pct_5_diff"] = (
    df["home_rolling_win_pct_5"] - df["away_rolling_win_pct_5"]
)

df["rolling_win_pct_10_diff"] = (
    df["home_rolling_win_pct_10"] - df["away_rolling_win_pct_10"]
)

df["rolling_point_diff_5_diff"] = (
    df["home_rolling_point_diff_5"] - df["away_rolling_point_diff_5"]
)

df["rolling_point_diff_10_diff"] = (
    df["home_rolling_point_diff_10"] - df["away_rolling_point_diff_10"]
)

df["rest_days_diff"] = df["home_rest_days"] - df["away_rest_days"]

In [83]:
feature_cols = [
    "home_rolling_win_pct_5",
    "away_rolling_win_pct_5",
    "home_rolling_win_pct_10",
    "away_rolling_win_pct_10",
    "home_rolling_point_diff_5",
    "away_rolling_point_diff_5",
    "home_rolling_point_diff_10",
    "away_rolling_point_diff_10",
    "home_rest_days",
    "away_rest_days",
    "home_elo",
    "away_elo",
    "elo_diff",
    "rolling_win_pct_5_diff",
    "rolling_win_pct_10_diff",
    "rolling_point_diff_5_diff",
    "rolling_point_diff_10_diff",
    "rest_days_diff",
]

df_model = df.dropna(subset=feature_cols).copy()

print("Model df shape:", df_model.shape)
df_model


Model df shape: (26358, 25)


,GAME_ID,GAME_DATE_EST,HOME_TEAM_ID,VISITOR_TEAM_ID,PTS_home,PTS_away,home_team_win,home_rolling_win_pct_5,home_rolling_win_pct_10,home_rolling_point_diff_5,...,away_rolling_point_diff_10,away_rest_days,home_elo,away_elo,elo_diff,rolling_win_pct_5_diff,rolling_win_pct_10_diff,rolling_point_diff_5_diff,rolling_point_diff_10_diff,rest_days_diff
141,20300127,2003-11-15,1610612764,1610612759,71.0,95.0,0,0.4,0.4,2.4,...,1.7,1.0,1479.349373,1495.319927,-15.970554,0.0,-0.1,-0.2,1.7,0.0
142,20300128,2003-11-15,1610612737,1610612751,85.0,100.0,0,0.4,0.3,-6.2,...,2.2,1.0,1479.059033,1498.465978,-19.406945,0.0,-0.2,-1.8,-6.1,2.0
144,20300130,2003-11-15,1610612739,1610612755,91.0,88.0,1,0.4,0.3,2.4,...,0.8,1.0,1482.054257,1508.740652,-26.686395,-0.2,-0.2,-0.4,-3.9,0.0
150,20300136,2003-11-16,1610612761,1610612745,101.0,97.0,1,0.2,0.5,-12.2,...,3.2,2.0,1498.859636,1517.696298,-18.836662,-0.6,-0.1,-19.2,-9.3,0.0
153,20300139,2003-11-17,1610612755,1610612745,66.0,74.0,0,0.6,0.5,3.6,...,5.2,1.0,1500.820016,1509.990779,-9.170763,0.0,-0.1,-0.4,-4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26518,22200474,2022-12-21,1610612760,1610612757,101.0,98.0,1,0.4,0.5,-0.6,...,3.4,2.0,1416.525781,1439.549888,-23.024107,-0.2,-0.1,-6.0,-4.4,0.0
26519,22200475,2022-12-21,1610612758,1610612747,134.0,120.0,1,0.4,0.6,-5.8,...,-2.6,2.0,1514.165040,1433.126182,81.038858,-0.2,0.1,-5.2,5.4,0.0
26520,22200476,2022-12-21,1610612746,1610612766,126.0,105.0,1,0.8,0.5,6.2,...,-7.7,2.0,1508.311444,1372.815989,135.495456,0.6,0.3,14.6,6.0,2.0
26521,22200477,2022-12-22,1610612740,1610612759,126.0,117.0,1,0.2,0.6,-6.4,...,-6.2,3.0,1563.074451,1379.702081,183.372370,-0.4,0.2,-6.0,9.8,0.0


In [80]:
model_df = df_model[
    [
        "GAME_ID",
        "GAME_DATE_EST",
        "HOME_TEAM_ID",
        "VISITOR_TEAM_ID",
        "home_team_win",
        "home_rolling_win_pct_5",
        "away_rolling_win_pct_5",
        "home_rolling_win_pct_10",
        "away_rolling_win_pct_10",
        "home_rolling_point_diff_5",
        "away_rolling_point_diff_5",
        "home_rolling_point_diff_10",
        "away_rolling_point_diff_10",
        "home_rest_days",
        "away_rest_days",
        "home_elo",
        "away_elo",
        "elo_diff",
        "rolling_win_pct_5_diff",
        "rolling_win_pct_10_diff",
        "rolling_point_diff_5_diff",
        "rolling_point_diff_10_diff",
        "rest_days_diff",
    ]
].copy()

print("Final model_df shape:", model_df.shape)
model_df.columns.tolist()

Final model_df shape: (26358, 23)


['GAME_ID',
 'GAME_DATE_EST',
 'HOME_TEAM_ID',
 'VISITOR_TEAM_ID',
 'home_team_win',
 'home_rolling_win_pct_5',
 'away_rolling_win_pct_5',
 'home_rolling_win_pct_10',
 'away_rolling_win_pct_10',
 'home_rolling_point_diff_5',
 'away_rolling_point_diff_5',
 'home_rolling_point_diff_10',
 'away_rolling_point_diff_10',
 'home_rest_days',
 'away_rest_days',
 'home_elo',
 'away_elo',
 'elo_diff',
 'rolling_win_pct_5_diff',
 'rolling_win_pct_10_diff',
 'rolling_point_diff_5_diff',
 'rolling_point_diff_10_diff',
 'rest_days_diff']

In [81]:
model_df

,GAME_ID,GAME_DATE_EST,HOME_TEAM_ID,VISITOR_TEAM_ID,home_team_win,home_rolling_win_pct_5,away_rolling_win_pct_5,home_rolling_win_pct_10,away_rolling_win_pct_10,home_rolling_point_diff_5,...,home_rest_days,away_rest_days,home_elo,away_elo,elo_diff,rolling_win_pct_5_diff,rolling_win_pct_10_diff,rolling_point_diff_5_diff,rolling_point_diff_10_diff,rest_days_diff
141,20300127,2003-11-15,1610612764,1610612759,0,0.4,0.4,0.4,0.5,2.4,...,1.0,1.0,1479.349373,1495.319927,-15.970554,0.0,-0.1,-0.2,1.7,0.0
142,20300128,2003-11-15,1610612737,1610612751,0,0.4,0.4,0.3,0.5,-6.2,...,3.0,1.0,1479.059033,1498.465978,-19.406945,0.0,-0.2,-1.8,-6.1,2.0
144,20300130,2003-11-15,1610612739,1610612755,1,0.4,0.6,0.3,0.5,2.4,...,1.0,1.0,1482.054257,1508.740652,-26.686395,-0.2,-0.2,-0.4,-3.9,0.0
150,20300136,2003-11-16,1610612761,1610612745,1,0.2,0.8,0.5,0.6,-12.2,...,2.0,2.0,1498.859636,1517.696298,-18.836662,-0.6,-0.1,-19.2,-9.3,0.0
153,20300139,2003-11-17,1610612755,1610612745,0,0.6,0.6,0.5,0.6,3.6,...,2.0,1.0,1500.820016,1509.990779,-9.170763,0.0,-0.1,-0.4,-4.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26518,22200474,2022-12-21,1610612760,1610612757,1,0.4,0.6,0.5,0.6,-0.6,...,2.0,2.0,1416.525781,1439.549888,-23.024107,-0.2,-0.1,-6.0,-4.4,0.0
26519,22200475,2022-12-21,1610612758,1610612747,1,0.4,0.6,0.6,0.5,-5.8,...,2.0,2.0,1514.165040,1433.126182,81.038858,-0.2,0.1,-5.2,5.4,0.0
26520,22200476,2022-12-21,1610612746,1610612766,1,0.8,0.2,0.5,0.2,6.2,...,4.0,2.0,1508.311444,1372.815989,135.495456,0.6,0.3,14.6,6.0,2.0
26521,22200477,2022-12-22,1610612740,1610612759,1,0.2,0.6,0.6,0.4,-6.4,...,3.0,3.0,1563.074451,1379.702081,183.372370,-0.4,0.2,-6.0,9.8,0.0
